SPDX-FileCopyrightText: 2026 Rodrigo López Guerra

SPDX-FileCopyrightText: 2026 Abigail Pérez García

SPDX-FileCopyrightText: 2026 Fernando Maggi Llerandi

SPDX-License-Identifier: Apache-2.0

In [3]:
import json, random, re, csv, time
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import vertexai
from vertexai.batch_prediction import BatchPredictionJob
from google.cloud import storage

PROJECT_ID    = "PROJECT-NAME"
LOCATION      = "us-central1"
BUCKET_NAME   = "BUCKET-NAME"
MODEL         = "gemini-2.5-flash"

RANDOM_SEED = 42

# IMPORTANTE:
# None = no truncar código.
# Esto reduce muchísimo los "inconclusive".
MAX_LINES = None

# Para validar todos los Type II/III/IV con LLM:
SAMPLE_RATIO_BY_TYPE = {
    1: 0.10,
    2: 0.30,
    3: 0.30,
    4: 0.35,
}

DATASETS = {
    "dataset solutions": {
        "clones": "./dataset solutions/clones",
        "js_files": "./dataset solutions/js_files",
    },
}

OUTPUT_FOLDER = Path("./validation")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

vertexai.init(project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(project=PROJECT_ID)

random.seed(RANDOM_SEED)

print("Config OK")
print("RUN_ID:", RUN_ID)
print("MAX_LINES:", MAX_LINES)
print("SAMPLE_RATIO_BY_TYPE:", SAMPLE_RATIO_BY_TYPE)

Config OK
RUN_ID: 20260518_120614
MAX_LINES: None
SAMPLE_RATIO_BY_TYPE: {1: 0.1, 2: 0.3, 3: 0.3, 4: 0.35}


In [58]:
TYPE_DESCRIPTIONS = {
    1: "Type I: identical except for whitespace and comments. The non-comment lexical token sequence must remain the same.",
    2: "Type II: same overall structure, control flow, statement order, algorithmic strategy and behavior, with user-defined identifiers renamed consistently.",
    3: "Type III: same behavior with moderate structural additions, deletions, splits, merges or rewrites; still recognizably related to the original.",
    4: "Type IV: same behavior through a substantially different implementation strategy or syntax.",
}

VALIDATION_PROMPT = '''You are a JavaScript code clone TYPE validator.

You receive:
- a COMPLETE ORIGINAL JavaScript program
- a COMPLETE CLONE JavaScript program
- a declared clone type

Your task:
Judge whether the CLONE is a valid example of the declared clone type.

Important evaluation philosophy:
- Do not be overly punitive.
- Use "major" only for clear, actionable clone-type violations.
- Use "minor" for borderline or small issues where the clone is still reasonably usable as the declared type.
- Treat "pass" and "minor" as valid.
- Use "inconclusive" only when the code explicitly contains the marker [TRUNCATED_FOR_LLM_VALIDATION].
- If there is no truncation marker, do not say the code is truncated, incomplete, or impossible to assess.

Behavior policy:
- You are not running the code.
- Do not reject for behavior unless there is a clear visible contradiction in the code.
- If output formatting, string contents, constants, loop bounds, or externally visible behavior clearly changed, mark it as major.
- If behavior appears preserved or cannot be disproven from the visible complete files, focus mainly on clone-type compliance.

Truncation rule:
- The provided code is complete unless it explicitly contains [TRUNCATED_FOR_LLM_VALIDATION].
- Only use severity "inconclusive" if [TRUNCATED_FOR_LLM_VALIDATION] is present.
- If there is no truncation marker, choose "pass", "minor", or "major".

Type-specific rules:

Type I:
- Valid only if the clone differs from the original only in whitespace and comments.
- All non-comment lexical tokens must be exactly the same and in the same order.
- Identifiers, literals, operators, punctuation, statement order, and executable code must not change.
- Note: Type I is normally validated locally, so you may not receive Type I examples.

Type II:
- Valid if the clone preserves the same overall structure, control flow, statement order, algorithmic strategy, and behavior.
- User-defined identifiers should be renamed consistently.
- Built-ins, standard globals, library APIs, DOM APIs, imports, exports, and behavior-critical external names must not be renamed.
- Do not reject solely because of harmless JavaScript surface changes such as:
  - var changed to let or const
  - quote style changed while string contents remain identical
  - numeric literals changed to equivalent representations such as 10 and 0xa
  - parseInt gains an explicit radix when behavior is unchanged
  - parentheses, semicolons, or formatting are adjusted
  - comments are added, removed, or rewritten
- Treat harmless surface changes as "minor" at most, not "major".
- Reject as "major" only when the clone clearly changes the structure, algorithmic strategy, statement order, behavior-critical properties, operators in a behavior-relevant way, or removes/adds meaningful executable logic.
- If a clone is mostly Type II but has one small harmless syntactic modernization, return valid true with severity "minor".

Type III:
- Valid if the clone preserves behavior and contains meaningful structural edits beyond pure renaming/reformatting.
- Meaningful structural edits include helper variables, helper functions, splitting expressions, merging statements, guard clauses, extracting constants, adding intermediate computation steps, or rewriting local expressions equivalently.
- It must remain recognizably related to the original.
- If the clone is mostly Type II but has small structural edits, return valid true with severity "minor" instead of "major".
- Reject as "major" only if there is clearly no meaningful structural edit, or if it becomes a completely different Type IV-style implementation.
- Do not reject merely because the changes are moderate rather than extensive.

Type IV:
- Valid if the clone shows a substantially different implementation strategy in the core logic, input processing, data representation, helper structure, or control-flow design.
- It does not need every line to be different.
- If the core logic is different but some boilerplate remains similar, return "pass" or "minor", not "major".
- Reject as "major" only if the clone is clearly just Type II or Type III-level editing with no substantial strategy change.
- If the implementation strategy is meaningfully different in the core algorithm, count it as Type IV even if some parsing/output boilerplate remains similar.

Decision rules:
- "pass": clearly valid clone type.
- "minor": valid clone with small or borderline concern.
- "major": clear clone-type violation or clear behavior violation.
- "inconclusive": only when [TRUNCATED_FOR_LLM_VALIDATION] is present.
- "minor" is still valid.
- Do not inflate "major" for harmless modernizations or style-level differences.

Respond ONLY with one JSON object, no markdown, no backticks, no explanation outside JSON.

Required JSON schema:
{
  "id": "<id>",
  "valid": true or false,
  "reason": "one short sentence explaining the main decision",
  "behavior_equivalent": true or false,
  "type_compliant": true or false,
  "severity": "pass" or "minor" or "major" or "inconclusive"
}
'''

print("Prompt OK")

Prompt OK


In [59]:
# Cell 3 — Funciones locales sin bloquear por Node behavior

import re
import time
from pathlib import Path

# ============================================================
# CODE READING
# ============================================================

def read_code(path):
    return Path(path).read_text(encoding="utf-8", errors="ignore")


def limit_code_text(text):
    """
    Aplica MAX_LINES solo si no es None.
    """
    if MAX_LINES is None:
        return text

    lines = text.splitlines()

    if len(lines) <= MAX_LINES:
        return text

    return "\n".join(lines[:MAX_LINES]) + "\n\n/* [TRUNCATED_FOR_LLM_VALIDATION] */"


def read_limited_code(path):
    return limit_code_text(read_code(path))


# ============================================================
# SANITY CHECKS
# ============================================================

def has_output_call(text):
    """
    Heurística: detecta salidas comunes.
    Solo se usa como warning, NO como fallo duro.
    """
    patterns = [
        r"\bconsole\.log\s*\(",
        r"\bconsole\.error\s*\(",
        r"\bprocess\.stdout\.write\s*\(",
        r"\bprocess\.stderr\.write\s*\(",
        r"\bfs\.writeSync\s*\(\s*1\s*,",
        r"\bfs\.writeSync\s*\(\s*2\s*,",
    ]

    return any(re.search(pattern, text) for pattern in patterns)


def uses_stdin(text):
    """
    Heurística: detecta si el programa probablemente usa stdin.
    Solo se usa como metadata para el LLM.
    """
    patterns = [
        r"fs\.readFileSync\s*\(\s*0",
        r"fs\.readFileSync\s*\(\s*['\"]\/dev\/stdin['\"]",
        r"process\.stdin",
        r"readline\.createInterface",
        r"\brl\.on\s*\(",
        r"\.on\s*\(\s*['\"]line['\"]",
        r"\.on\s*\(\s*['\"]close['\"]",
    ]

    return any(re.search(pattern, text) for pattern in patterns)


def quick_file_sanity(path):
    """
    Sanity check mínimo.

    Fallo duro solamente:
    - archivo faltante
    - archivo vacío

    Todo lo demás queda como warning.
    """
    path = Path(path)

    if not path.exists():
        return False, "missing_file", ""

    text = path.read_text(encoding="utf-8", errors="ignore")

    if not text.strip():
        return False, "empty_file", ""

    lines = text.splitlines()
    last_line = lines[-1].strip() if lines else ""

    suspicious_endings = [
        "function",
        "return",
        "if",
        "else",
        "for",
        "while",
        "=>",
        ".",
        ",",
        "=",
        "+",
        "-",
        "{",
        "(",
        "[",
    ]

    if any(last_line.endswith(x) for x in suspicious_endings):
        return True, "", f"suspicious_last_line: {last_line[:160]}"

    return True, "", ""


def static_sanity_check_from_text(original_path, clone_path, original_text, clone_text):
    """
    Devuelve:
    - sanity_ok
    - hard_reason
    - warning

    Solo bloquea archivos faltantes o vacíos.
    """
    warnings = []

    orig_ok, orig_reason, orig_warning = quick_file_sanity(original_path)

    if not orig_ok:
        return False, f"Original appears invalid locally: {orig_reason}", ""

    if orig_warning:
        warnings.append(f"original_warning={orig_warning}")

    clone_ok, clone_reason, clone_warning = quick_file_sanity(clone_path)

    if not clone_ok:
        return False, f"Clone appears invalid locally: {clone_reason}", ""

    if clone_warning:
        warnings.append(f"clone_warning={clone_warning}")

    if has_output_call(original_text) and not has_output_call(clone_text):
        warnings.append("original_has_output_calls_but_clone_has_no_common_output_call")

    return True, "", " | ".join(warnings)


# ============================================================
# TYPE I TOKEN EQUIVALENCE
# ============================================================

def normalize_js_for_type1(code):
    """
    Quita whitespace y comentarios fuera de strings/templates.
    Sirve para validar Type I localmente.
    """
    out = []
    i = 0
    n = len(code)

    state = "normal"
    quote = None

    while i < n:
        ch = code[i]
        nxt = code[i + 1] if i + 1 < n else ""

        if state == "normal":
            # whitespace
            if ch.isspace():
                i += 1
                continue

            # line comment
            if ch == "/" and nxt == "/":
                i += 2
                while i < n and code[i] not in "\r\n":
                    i += 1
                continue

            # block comment
            if ch == "/" and nxt == "*":
                i += 2
                while i + 1 < n and not (code[i] == "*" and code[i + 1] == "/"):
                    i += 1
                i += 2
                continue

            # string/template start
            if ch in ("'", '"', "`"):
                state = "string"
                quote = ch
                out.append(ch)
                i += 1
                continue

            out.append(ch)
            i += 1
            continue

        if state == "string":
            out.append(ch)

            if ch == "\\":
                if i + 1 < n:
                    out.append(code[i + 1])
                    i += 2
                    continue

            if ch == quote:
                state = "normal"
                quote = None

            i += 1
            continue

    return "".join(out)


def type1_tokens_equivalent(original_code, clone_code):
    return normalize_js_for_type1(original_code) == normalize_js_for_type1(clone_code)


# ============================================================
# ID HELPERS
# ============================================================

def infer_from_id(custom_id):
    """
    Extrae dataset, file_id y clone_type desde IDs tipo:
    dataset_solutions|2987_clon_3.js
    """
    result = {
        "dataset": "",
        "file_id": "",
        "clone_type": None,
    }

    if "|" in str(custom_id):
        left, right = str(custom_id).split("|", 1)
        result["dataset"] = left.replace("_", " ")

        m = re.match(r"^(.+)_clon_(\d+)\.js$", right)
        if m:
            result["file_id"] = m.group(1)
            result["clone_type"] = int(m.group(2))

    return result


print("Funciones locales OK")

Funciones locales OK


In [60]:
# Cell 4 — Preparar JSONL sin local_node blocking

requests = []
meta = []
local_results = []
scan_stats = {}

original_code_cache = {}

PROGRESS_EVERY = 500

start_time = time.time()

for dataset_name, paths in DATASETS.items():
    clones_dir  = Path(paths["clones"])
    jsfiles_dir = Path(paths["js_files"])

    scan_stats[dataset_name] = {
        "total_clones": 0,
        "missing_original": 0,
        "bad_filename": 0,
        "local_sanity_hard_fail": 0,
        "local_sanity_warning": 0,
        "local_type1_pass": 0,
        "local_type1_fail": 0,
        "unsampled": 0,
        "llm_requests": 0,
    }

    if not clones_dir.exists():
        print(f"[WARN] No existe: {clones_dir}")
        continue

    clone_files = sorted(clones_dir.glob("*_clon_*.js"))
    scan_stats[dataset_name]["total_clones"] = len(clone_files)

    print(f"\n{dataset_name}: {len(clone_files)} clones encontrados")

    for idx, clone_path in enumerate(clone_files, start=1):
        if idx % PROGRESS_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"[{dataset_name}] {idx}/{len(clone_files)} procesados "
                f"| LLM: {scan_stats[dataset_name]['llm_requests']} "
                f"| local: {len(local_results)} "
                f"| elapsed: {elapsed/60:.1f} min"
            )

        m = re.match(r"^(.+)_clon_(\d+)\.js$", clone_path.name)

        if not m:
            scan_stats[dataset_name]["bad_filename"] += 1

            local_results.append({
                "id": f"{dataset_name.replace(' ', '_')}|{clone_path.name}",
                "dataset": dataset_name,
                "file_id": "",
                "clone_type": None,
                "valid": False,
                "reason": "Bad clone filename format.",
                "behavior_equivalent": None,
                "type_compliant": False,
                "validation_source": "local_bad_filename",
                "severity": "major",
                "local_sanity_warning": "",
            })

            continue

        file_id = m.group(1)
        clone_type = int(m.group(2))

        orig_path = jsfiles_dir / f"{file_id}.js"
        custom_id = f"{dataset_name.replace(' ', '_')}|{clone_path.name}"

        if not orig_path.exists():
            scan_stats[dataset_name]["missing_original"] += 1

            local_results.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "valid": False,
                "reason": f"Missing original file: {orig_path}",
                "behavior_equivalent": None,
                "type_compliant": False,
                "validation_source": "local_missing_original",
                "severity": "major",
                "local_sanity_warning": "",
            })

            continue

        # ------------------------------------------------------------
        # Leer código
        # ------------------------------------------------------------

        orig_key = str(orig_path.resolve())

        if orig_key not in original_code_cache:
            original_code_cache[orig_key] = read_code(orig_path)

        original_code = original_code_cache[orig_key]
        clone_code = read_code(clone_path)

        original_uses_stdin = uses_stdin(original_code)
        clone_uses_stdin = uses_stdin(clone_code)

        # ------------------------------------------------------------
        # Sanity mínimo: solo bloquea missing/empty
        # ------------------------------------------------------------

        sanity_ok, sanity_hard_reason, sanity_warning = static_sanity_check_from_text(
            orig_path,
            clone_path,
            original_code,
            clone_code
        )

        if not sanity_ok:
            scan_stats[dataset_name]["local_sanity_hard_fail"] += 1

            local_results.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "valid": False,
                "reason": sanity_hard_reason,
                "behavior_equivalent": None,
                "type_compliant": False,
                "validation_source": "local_static_sanity",
                "severity": "major",
                "local_sanity_warning": "",
            })

            continue

        if sanity_warning:
            scan_stats[dataset_name]["local_sanity_warning"] += 1

        # ------------------------------------------------------------
        # Type I: local token-equivalence
        # IMPORTANTE: no usamos Node behavior aquí.
        # ------------------------------------------------------------

        if clone_type == 1:
            if type1_tokens_equivalent(original_code, clone_code):
                scan_stats[dataset_name]["local_type1_pass"] += 1

                local_results.append({
                    "id": custom_id,
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_type": clone_type,
                    "valid": True,
                    "reason": "Type I token-equivalence passed locally.",
                    "behavior_equivalent": True,
                    "type_compliant": True,
                    "validation_source": "local_type1",
                    "severity": "pass",
                    "local_sanity_warning": sanity_warning,
                })

            else:
                scan_stats[dataset_name]["local_type1_fail"] += 1

                local_results.append({
                    "id": custom_id,
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_type": clone_type,
                    "valid": False,
                    "reason": "Type I token-equivalence failed after removing comments and whitespace.",
                    "behavior_equivalent": None,
                    "type_compliant": False,
                    "validation_source": "local_type1",
                    "severity": "major",
                    "local_sanity_warning": sanity_warning,
                })

            continue

        # ------------------------------------------------------------
        # Type II / III / IV: mandar al LLM según SAMPLE_RATIO
        # IMPORTANTE: no bloqueamos por Node con stdin vacío.
        # ------------------------------------------------------------

        ratio = SAMPLE_RATIO_BY_TYPE.get(clone_type, 0.10)
        send_to_llm = random.random() < ratio

        if not send_to_llm:
            scan_stats[dataset_name]["unsampled"] += 1

            local_results.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "valid": None,
                "reason": "Not sampled for LLM type validation.",
                "behavior_equivalent": None,
                "type_compliant": None,
                "validation_source": "local_unsampled",
                "severity": "not_audited",
                "local_sanity_warning": sanity_warning,
            })

            continue

        type_desc = TYPE_DESCRIPTIONS[clone_type]

        original_code_for_llm = limit_code_text(original_code)
        clone_code_for_llm = limit_code_text(clone_code)

        original_truncated = "[TRUNCATED_FOR_LLM_VALIDATION]" in original_code_for_llm
        clone_truncated = "[TRUNCATED_FOR_LLM_VALIDATION]" in clone_code_for_llm

        original_line_count = len(original_code_for_llm.splitlines())
        clone_line_count = len(clone_code_for_llm.splitlines())

        prompt_text = (
            VALIDATION_PROMPT
            + f"\n\nPair ID: {custom_id}"
            + f"\nDeclared clone type: {clone_type} - {type_desc}"
            + f"\nOriginal line count: {original_line_count}"
            + f"\nClone line count: {clone_line_count}"
            + f"\nOriginal truncated: {original_truncated}"
            + f"\nClone truncated: {clone_truncated}"
            + f"\nOriginal uses stdin: {original_uses_stdin}"
            + f"\nClone uses stdin: {clone_uses_stdin}"
            + f"\nLocal sanity warning: {sanity_warning if sanity_warning else 'none'}"
            + "\nFiles are complete unless truncated is true."
            + "\nImportant: This validator is focused on clone-type compliance. Do not assume empty-stdin behavior is a definitive test."
            + f"\n\n[ORIGINAL_COMPLETE_FILE_START]\n{original_code_for_llm}\n[ORIGINAL_COMPLETE_FILE_END]"
            + f"\n\n[CLONE_COMPLETE_FILE_START]\n{clone_code_for_llm}\n[CLONE_COMPLETE_FILE_END]"
        )

        requests.append({
            "custom_id": custom_id,
            "request": {
                "model": f"models/{MODEL}",
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": prompt_text}],
                    }
                ],
                "generation_config": {
                    "temperature": 0,
                },
            },
        })

        meta.append({
            "id": custom_id,
            "dataset": dataset_name,
            "file_id": file_id,
            "clone_type": clone_type,
            "original_uses_stdin": original_uses_stdin,
            "clone_uses_stdin": clone_uses_stdin,
            "local_sanity_warning": sanity_warning,
        })

        scan_stats[dataset_name]["llm_requests"] += 1

# ============================================================
# GUARDAR ARCHIVOS
# ============================================================

jsonl_path = Path("validation_requests.jsonl")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for req in requests:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

with open(OUTPUT_FOLDER / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

with open(OUTPUT_FOLDER / "local_results.json", "w", encoding="utf-8") as f:
    json.dump(local_results, f, ensure_ascii=False, indent=2)

with open(OUTPUT_FOLDER / "scan_stats.json", "w", encoding="utf-8") as f:
    json.dump(scan_stats, f, ensure_ascii=False, indent=2)

elapsed = time.time() - start_time

print("\nPreparación terminada")
print("Requests LLM:", len(requests))
print("Local results:", len(local_results))
print("JSONL:", jsonl_path)
print(f"Tiempo total: {elapsed/60:.2f} min")

print("\nScan stats:")
print(json.dumps(scan_stats, indent=2, ensure_ascii=False))


dataset solutions: 19996 clones encontrados
[dataset solutions] 500/19996 procesados | LLM: 131 | local: 368 | elapsed: 0.0 min
[dataset solutions] 1000/19996 procesados | LLM: 250 | local: 749 | elapsed: 0.0 min
[dataset solutions] 1500/19996 procesados | LLM: 353 | local: 1146 | elapsed: 0.0 min
[dataset solutions] 2000/19996 procesados | LLM: 469 | local: 1530 | elapsed: 0.0 min
[dataset solutions] 2500/19996 procesados | LLM: 577 | local: 1922 | elapsed: 0.0 min
[dataset solutions] 3000/19996 procesados | LLM: 697 | local: 2302 | elapsed: 0.0 min
[dataset solutions] 3500/19996 procesados | LLM: 812 | local: 2687 | elapsed: 0.0 min
[dataset solutions] 4000/19996 procesados | LLM: 939 | local: 3060 | elapsed: 0.0 min
[dataset solutions] 4500/19996 procesados | LLM: 1060 | local: 3439 | elapsed: 0.0 min
[dataset solutions] 5000/19996 procesados | LLM: 1179 | local: 3820 | elapsed: 0.0 min
[dataset solutions] 5500/19996 procesados | LLM: 1296 | local: 4203 | elapsed: 0.0 min
[dataset 

In [61]:
hard_fails = [
    r for r in local_results
    if r.get("validation_source") == "local_static_sanity"
]

print("Hard fails:", len(hard_fails))

for r in hard_fails:
    print(
        f"- type={r.get('clone_type')} "
        f"| id={r.get('id')} "
        f"| reason={r.get('reason')}"
    )

Hard fails: 0


In [62]:
sanity_reasons = Counter(
    r["reason"] for r in local_results
    if r.get("validation_source") == "local_static_sanity"
)

print("Top reasons SOLO local_static_sanity:")
for reason, count in sanity_reasons.most_common(30):
    print(f"{count}: {reason}")

Top reasons SOLO local_static_sanity:


In [63]:
behavior_reasons = Counter(
    r["reason"] for r in local_results
    if r.get("validation_source") == "local_node"
)

print("Top reasons SOLO local_node:")
for reason, count in behavior_reasons.most_common(30):
    print(f"{count}: {reason}")

Top reasons SOLO local_node:


In [64]:
from collections import Counter

reasons = Counter(r["reason"] for r in local_results)

print("Top reasons local_sanity_fail:")
for reason, count in reasons.most_common(20):
    print(f"{count}: {reason}")

Top reasons local_sanity_fail:
10282: Not sampled for LLM type validation.
4999: Type I token-equivalence passed locally.


In [65]:
# Cell 4 — Preparar JSONL sin local_node blocking

requests = []
meta = []
local_results = []
scan_stats = {}

original_code_cache = {}

PROGRESS_EVERY = 500

start_time = time.time()

for dataset_name, paths in DATASETS.items():
    clones_dir  = Path(paths["clones"])
    jsfiles_dir = Path(paths["js_files"])

    scan_stats[dataset_name] = {
        "total_clones": 0,
        "missing_original": 0,
        "bad_filename": 0,
        "local_sanity_hard_fail": 0,
        "local_sanity_warning": 0,
        "local_type1_pass": 0,
        "local_type1_fail": 0,
        "unsampled": 0,
        "llm_requests": 0,
    }

    if not clones_dir.exists():
        print(f"[WARN] No existe: {clones_dir}")
        continue

    clone_files = sorted(clones_dir.glob("*_clon_*.js"))
    scan_stats[dataset_name]["total_clones"] = len(clone_files)

    print(f"\n{dataset_name}: {len(clone_files)} clones encontrados")

    for idx, clone_path in enumerate(clone_files, start=1):
        if idx % PROGRESS_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"[{dataset_name}] {idx}/{len(clone_files)} procesados "
                f"| LLM: {scan_stats[dataset_name]['llm_requests']} "
                f"| local: {len(local_results)} "
                f"| elapsed: {elapsed/60:.1f} min"
            )

        m = re.match(r"^(.+)_clon_(\d+)\.js$", clone_path.name)

        if not m:
            scan_stats[dataset_name]["bad_filename"] += 1

            local_results.append({
                "id": f"{dataset_name.replace(' ', '_')}|{clone_path.name}",
                "dataset": dataset_name,
                "file_id": "",
                "clone_type": None,
                "valid": False,
                "reason": "Bad clone filename format.",
                "behavior_equivalent": None,
                "type_compliant": False,
                "validation_source": "local_bad_filename",
                "severity": "major",
                "local_sanity_warning": "",
            })

            continue

        file_id = m.group(1)
        clone_type = int(m.group(2))

        orig_path = jsfiles_dir / f"{file_id}.js"
        custom_id = f"{dataset_name.replace(' ', '_')}|{clone_path.name}"

        if not orig_path.exists():
            scan_stats[dataset_name]["missing_original"] += 1

            local_results.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "valid": False,
                "reason": f"Missing original file: {orig_path}",
                "behavior_equivalent": None,
                "type_compliant": False,
                "validation_source": "local_missing_original",
                "severity": "major",
                "local_sanity_warning": "",
            })

            continue

        # ------------------------------------------------------------
        # Leer código
        # ------------------------------------------------------------

        orig_key = str(orig_path.resolve())

        if orig_key not in original_code_cache:
            original_code_cache[orig_key] = read_code(orig_path)

        original_code = original_code_cache[orig_key]
        clone_code = read_code(clone_path)

        original_uses_stdin = uses_stdin(original_code)
        clone_uses_stdin = uses_stdin(clone_code)

        # ------------------------------------------------------------
        # Sanity mínimo: solo bloquea missing/empty
        # ------------------------------------------------------------

        sanity_ok, sanity_hard_reason, sanity_warning = static_sanity_check_from_text(
            orig_path,
            clone_path,
            original_code,
            clone_code
        )

        if not sanity_ok:
            scan_stats[dataset_name]["local_sanity_hard_fail"] += 1

            local_results.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "valid": False,
                "reason": sanity_hard_reason,
                "behavior_equivalent": None,
                "type_compliant": False,
                "validation_source": "local_static_sanity",
                "severity": "major",
                "local_sanity_warning": "",
            })

            continue

        if sanity_warning:
            scan_stats[dataset_name]["local_sanity_warning"] += 1

        # ------------------------------------------------------------
        # Type I: local token-equivalence
        # IMPORTANTE: no usamos Node behavior aquí.
        # ------------------------------------------------------------

        if clone_type == 1:
            if type1_tokens_equivalent(original_code, clone_code):
                scan_stats[dataset_name]["local_type1_pass"] += 1

                local_results.append({
                    "id": custom_id,
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_type": clone_type,
                    "valid": True,
                    "reason": "Type I token-equivalence passed locally.",
                    "behavior_equivalent": True,
                    "type_compliant": True,
                    "validation_source": "local_type1",
                    "severity": "pass",
                    "local_sanity_warning": sanity_warning,
                })

            else:
                scan_stats[dataset_name]["local_type1_fail"] += 1

                local_results.append({
                    "id": custom_id,
                    "dataset": dataset_name,
                    "file_id": file_id,
                    "clone_type": clone_type,
                    "valid": False,
                    "reason": "Type I token-equivalence failed after removing comments and whitespace.",
                    "behavior_equivalent": None,
                    "type_compliant": False,
                    "validation_source": "local_type1",
                    "severity": "major",
                    "local_sanity_warning": sanity_warning,
                })

            continue

        # ------------------------------------------------------------
        # Type II / III / IV: mandar al LLM según SAMPLE_RATIO
        # IMPORTANTE: no bloqueamos por Node con stdin vacío.
        # ------------------------------------------------------------

        ratio = SAMPLE_RATIO_BY_TYPE.get(clone_type, 0.10)
        send_to_llm = random.random() < ratio

        if not send_to_llm:
            scan_stats[dataset_name]["unsampled"] += 1

            local_results.append({
                "id": custom_id,
                "dataset": dataset_name,
                "file_id": file_id,
                "clone_type": clone_type,
                "valid": None,
                "reason": "Not sampled for LLM type validation.",
                "behavior_equivalent": None,
                "type_compliant": None,
                "validation_source": "local_unsampled",
                "severity": "not_audited",
                "local_sanity_warning": sanity_warning,
            })

            continue

        type_desc = TYPE_DESCRIPTIONS[clone_type]

        original_code_for_llm = limit_code_text(original_code)
        clone_code_for_llm = limit_code_text(clone_code)

        original_truncated = "[TRUNCATED_FOR_LLM_VALIDATION]" in original_code_for_llm
        clone_truncated = "[TRUNCATED_FOR_LLM_VALIDATION]" in clone_code_for_llm

        original_line_count = len(original_code_for_llm.splitlines())
        clone_line_count = len(clone_code_for_llm.splitlines())

        prompt_text = (
            VALIDATION_PROMPT
            + f"\n\nPair ID: {custom_id}"
            + f"\nDeclared clone type: {clone_type} - {type_desc}"
            + f"\nOriginal line count: {original_line_count}"
            + f"\nClone line count: {clone_line_count}"
            + f"\nOriginal truncated: {original_truncated}"
            + f"\nClone truncated: {clone_truncated}"
            + f"\nOriginal uses stdin: {original_uses_stdin}"
            + f"\nClone uses stdin: {clone_uses_stdin}"
            + "\nFiles are complete unless truncated is true."
            + "\nImportant: This validator is focused on clone-type compliance. Do not assume empty-stdin behavior is a definitive test."
            + f"\n\n[ORIGINAL_COMPLETE_FILE_START]\n{original_code_for_llm}\n[ORIGINAL_COMPLETE_FILE_END]"
            + f"\n\n[CLONE_COMPLETE_FILE_START]\n{clone_code_for_llm}\n[CLONE_COMPLETE_FILE_END]"
        )

        requests.append({
            "custom_id": custom_id,
            "request": {
                "model": f"models/{MODEL}",
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": prompt_text}]
                    }
                ],
                "generation_config": {
                    "temperature": 0
                },
            }
        })

        meta.append({
            "id": custom_id,
            "dataset": dataset_name,
            "file_id": file_id,
            "clone_type": clone_type,
            "original_uses_stdin": original_uses_stdin,
            "clone_uses_stdin": clone_uses_stdin,
            "local_sanity_warning": sanity_warning,
        })

        scan_stats[dataset_name]["llm_requests"] += 1

# ============================================================
# GUARDAR ARCHIVOS
# ============================================================

jsonl_path = Path("validation_requests.jsonl")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for req in requests:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

with open(OUTPUT_FOLDER / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

with open(OUTPUT_FOLDER / "local_results.json", "w", encoding="utf-8") as f:
    json.dump(local_results, f, ensure_ascii=False, indent=2)

with open(OUTPUT_FOLDER / "scan_stats.json", "w", encoding="utf-8") as f:
    json.dump(scan_stats, f, ensure_ascii=False, indent=2)

elapsed = time.time() - start_time

print("\nPreparación terminada")
print("Requests LLM:", len(requests))
print("Local results:", len(local_results))
print("JSONL:", jsonl_path)
print(f"Tiempo total: {elapsed/60:.2f} min")

print("\nScan stats:")
print(json.dumps(scan_stats, indent=2, ensure_ascii=False))


dataset solutions: 19996 clones encontrados
[dataset solutions] 500/19996 procesados | LLM: 102 | local: 397 | elapsed: 0.0 min
[dataset solutions] 1000/19996 procesados | LLM: 216 | local: 783 | elapsed: 0.0 min
[dataset solutions] 1500/19996 procesados | LLM: 346 | local: 1153 | elapsed: 0.0 min
[dataset solutions] 2000/19996 procesados | LLM: 455 | local: 1544 | elapsed: 0.0 min
[dataset solutions] 2500/19996 procesados | LLM: 575 | local: 1924 | elapsed: 0.0 min
[dataset solutions] 3000/19996 procesados | LLM: 699 | local: 2300 | elapsed: 0.0 min
[dataset solutions] 3500/19996 procesados | LLM: 826 | local: 2673 | elapsed: 0.0 min
[dataset solutions] 4000/19996 procesados | LLM: 937 | local: 3062 | elapsed: 0.0 min
[dataset solutions] 4500/19996 procesados | LLM: 1070 | local: 3429 | elapsed: 0.0 min
[dataset solutions] 5000/19996 procesados | LLM: 1179 | local: 3820 | elapsed: 0.0 min
[dataset solutions] 5500/19996 procesados | LLM: 1281 | local: 4218 | elapsed: 0.0 min
[dataset 

In [ ]:
def upload_jsonl_to_gcs(jsonl_path):
    bucket = storage_client.bucket(BUCKET_NAME)

    blob_name = f"validation/input/run_{RUN_ID}/validation_requests.jsonl"
    blob = bucket.blob(blob_name)

    blob.upload_from_filename(
        str(jsonl_path),
        content_type="application/jsonl",
    )

    return f"gs://{BUCKET_NAME}/{blob_name}"


def submit_validation_batch(jsonl_path):
    input_uri = upload_jsonl_to_gcs(jsonl_path)
    output_uri = f"gs://{BUCKET_NAME}/validation/output/run_{RUN_ID}/"

    print("Input :", input_uri)
    print("Output:", output_uri)
    print("Submitting validation batch...")

    job = BatchPredictionJob.submit(
        source_model=MODEL,
        input_dataset=input_uri,
        output_uri_prefix=output_uri,
    )

    print("\nBatch submitted")
    print("Job name:", job.resource_name)
    print("State:", job.state)

    return job.resource_name, output_uri


if len(requests) == 0:
    print("No hay requests LLM. No se lanza batch.")
    job_name = None
    output_uri = None
else:
    job_name, output_uri = submit_validation_batch(jsonl_path)

with open(OUTPUT_FOLDER / "batch_info.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "job_name": job_name,
            "output_uri": output_uri,
            "run_id": RUN_ID,
            "jsonl_path": str(jsonl_path),
            "num_requests": len(requests),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

In [70]:
def refresh_job(job_name):
    if not job_name:
        print("No hay job_name.")
        return None

    job = BatchPredictionJob(job_name)
    job.refresh()

    print("Job:", job.resource_name)
    print("State:", job.state)

    return job


try:
    job_name
except NameError:
    batch_info_path = OUTPUT_FOLDER / "batch_info.json"
    with open(batch_info_path, "r", encoding="utf-8") as f:
        batch_info = json.load(f)
    job_name = batch_info["job_name"]
    output_uri = batch_info["output_uri"]

job = refresh_job(job_name)

Job: projects/753506274754/locations/us-central1/batchPredictionJobs/8342842151343751168
State: 4


In [71]:
def extract_json_object(text):
    """
    Extrae JSON aunque venga con ruido mínimo.
    """
    if text is None:
        raise ValueError("empty_text")

    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("no_json_object_found")

    return json.loads(m.group(0))


def parse_batch_line(data):
    custom_id = data.get("custom_id") or data.get("id") or "unknown"

    if "error" in data:
        return {
            "id": custom_id,
            "valid": None,
            "reason": f"model_error: {data['error']}",
            "behavior_equivalent": None,
            "type_compliant": None,
            "severity": "parse_or_model_error",
            "validation_source": "llm",
        }

    try:
        text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
        parsed = extract_json_object(text)

        # Si el modelo omitió o cambió id, usamos custom_id.
        parsed["id"] = custom_id if custom_id != "unknown" else parsed.get("id", "unknown")

        parsed["validation_source"] = "llm"
        return parsed

    except Exception as e:
        return {
            "id": custom_id,
            "valid": None,
            "reason": f"parse_error: {e}",
            "behavior_equivalent": None,
            "type_compliant": None,
            "severity": "parse_or_model_error",
            "validation_source": "llm",
        }


def download_and_parse_llm_results(output_uri):
    if not output_uri:
        print("No hay output_uri.")
        return []

    bucket = storage_client.bucket(BUCKET_NAME)
    prefix = output_uri.replace(f"gs://{BUCKET_NAME}/", "")

    blobs = list(bucket.list_blobs(prefix=prefix))

    llm_results = []

    print("Blobs encontrados:", len(blobs))

    for blob in blobs:
        if not blob.name.endswith(".jsonl"):
            continue

        print("Leyendo:", blob.name)
        content = blob.download_as_text()

        for line in content.splitlines():
            if not line.strip():
                continue

            try:
                data = json.loads(line)
                parsed = parse_batch_line(data)
                llm_results.append(parsed)
            except Exception as e:
                llm_results.append({
                    "id": "unknown",
                    "valid": None,
                    "reason": f"line_parse_error: {e}",
                    "behavior_equivalent": None,
                    "type_compliant": None,
                    "severity": "parse_or_model_error",
                    "validation_source": "llm",
                })

    return llm_results


try:
    output_uri
except NameError:
    batch_info_path = OUTPUT_FOLDER / "batch_info.json"
    with open(batch_info_path, "r", encoding="utf-8") as f:
        batch_info = json.load(f)
    output_uri = batch_info["output_uri"]

llm_results = download_and_parse_llm_results(output_uri)

# Enriquecer con meta
meta_path = OUTPUT_FOLDER / "meta.json"
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

meta_by_id = {m["id"]: m for m in meta}

normalized_llm_results = []

for r in llm_results:
    rid = r.get("id", "")

    inferred = infer_from_id(rid)
    m = meta_by_id.get(rid, {})

    dataset = r.get("dataset") or m.get("dataset") or inferred.get("dataset")
    file_id = r.get("file_id") or m.get("file_id") or inferred.get("file_id")
    clone_type = r.get("clone_type") or m.get("clone_type") or inferred.get("clone_type")

    normalized = {
        "id": rid,
        "dataset": dataset,
        "file_id": file_id,
        "clone_type": clone_type,
        "valid": r.get("valid"),
        "reason": r.get("reason", ""),
        "behavior_equivalent": r.get("behavior_equivalent"),
        "type_compliant": r.get("type_compliant"),
        "severity": r.get("severity", ""),
        "validation_source": r.get("validation_source", "llm"),
        "original_uses_stdin": m.get("original_uses_stdin"),
        "clone_uses_stdin": m.get("clone_uses_stdin"),
        "local_sanity_warning": m.get("local_sanity_warning", ""),
    }

    normalized_llm_results.append(normalized)

with open(OUTPUT_FOLDER / "llm_results.json", "w", encoding="utf-8") as f:
    json.dump(normalized_llm_results, f, ensure_ascii=False, indent=2)

print("LLM results:", len(normalized_llm_results))
print("Guardado:", OUTPUT_FOLDER / "llm_results.json")

Blobs encontrados: 1
Leyendo: validation/output/run_20260518_025130/prediction-model-2026-05-18T08:52:49.154523Z/predictions.jsonl
LLM results: 4768
Guardado: validation/llm_results.json


In [1]:
# Cell — Convertir validation/llm_results.json a CSV

import json
import pandas as pd
from pathlib import Path

json_path = Path("validation/llm_results.json")
csv_path = Path("validation/llm_results.csv")

if not json_path.exists():
    raise FileNotFoundError(f"No existe: {json_path}")

def load_json_or_jsonl(path):
    """
    Carga JSON normal o JSONL.
    Devuelve una lista de registros.
    """
    text = path.read_text(encoding="utf-8").strip()

    if not text:
        return []

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        # Fallback por si realmente es JSONL
        rows = []
        for line in text.splitlines():
            line = line.strip()
            if line:
                rows.append(json.loads(line))
        return rows

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        # Si el JSON tiene una llave principal con lista adentro
        for key in ["results", "rows", "data", "items", "records"]:
            if key in data and isinstance(data[key], list):
                return data[key]

        # Si es un solo objeto
        return [data]

    return [{"value": data}]

rows = load_json_or_jsonl(json_path)

df = pd.json_normalize(rows, sep="_")

# Reordenar columnas importantes si existen
preferred_cols = [
    "id",
    "custom_id",
    "dataset",
    "file_id",
    "clone_type",
    "clone_path",
    "status",
    "final_status",
    "valid",
    "invalid",
    "reason",
    "validation_source",
]

existing_preferred = [c for c in preferred_cols if c in df.columns]
other_cols = [c for c in df.columns if c not in existing_preferred]

df = df[existing_preferred + other_cols]

csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False, encoding="utf-8")

print("Guardado:", csv_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head(20))

Guardado: validation/llm_results.csv
Rows: 4760
Columns: 13


,id,dataset,file_id,clone_type,valid,reason,validation_source,behavior_equivalent,type_compliant,severity,original_uses_stdin,clone_uses_stdin,local_sanity_warning
0,dataset_solutions|4173_clon_2.js,dataset solutions,4173,2,True,"The clone preserves the overall structure, con...",llm,True,True,pass,False,False,
1,dataset_solutions|2431_clon_2.js,dataset solutions,2431,2,True,The clone preserves the same overall structure...,llm,True,True,pass,False,False,
2,dataset_solutions|1162_clon_3.js,dataset solutions,1162,3,True,The clone preserves behavior and introduces me...,llm,True,True,pass,True,True,
3,dataset_solutions|2159_clon_3.js,dataset solutions,2159,3,True,"The clone exhibits extensive structural edits,...",llm,True,True,pass,False,False,
4,dataset_solutions|2703_clon_3.js,dataset solutions,2703,3,True,The clone introduces helper functions for inpu...,llm,True,True,pass,False,False,
5,dataset_solutions|4113_clon_4.js,dataset solutions,4113,4,False,The clone rephrases the same arithmetic calcul...,llm,True,False,major,False,False,
6,dataset_solutions|4815_clon_2.js,dataset solutions,4815,2,True,"The clone preserves the overall structure, con...",llm,True,True,pass,False,False,
7,dataset_solutions|2330_clon_4.js,dataset solutions,2330,4,True,The clone uses a functional reduce method for ...,llm,True,True,pass,False,False,
8,dataset_solutions|2832_clon_3.js,dataset solutions,2832,3,True,"The clone refactors input processing, array in...",llm,True,True,pass,False,False,
9,dataset_solutions|3179_clon_2.js,dataset solutions,3179,2,True,"The clone preserves the overall structure, con...",llm,True,True,minor,False,False,


In [4]:
def norm(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower()


def final_status(row):
    severity = norm(row.get("severity"))
    valid = norm(row.get("valid"))
    behavior = norm(row.get("behavior_equivalent"))
    type_ok = norm(row.get("type_compliant"))
    reason = norm(row.get("reason"))
    source = norm(row.get("validation_source"))

    if severity == "not_audited":
        return "not_audited"

    # Local hard failures.
    if source in {"local_static_sanity", "local_missing_original", "local_bad_filename"}:
        return "invalid_local_file"

    # Errores del modelo/parsing.
    if severity in {"parse_or_model_error", "model_error", "parse_error"}:
        return "validation_error_retry"

    if "parse_error" in reason or "model_error" in reason:
        return "validation_error_retry"

    # Inconclusos o evidencia insuficiente.
    if severity == "inconclusive":
        return "needs_revalidation"

    if any(term in reason for term in [
        "truncated",
        "not enough visible evidence",
        "visible portion",
        "full assessment impossible",
        "impossible to assess",
        "impossible to verify",
    ]):
        return "needs_revalidation"

    # Major sí es inválido accionable.
    if severity == "major":
        if behavior in {"false", "incorrecto", "0", "no"}:
            return "invalid_behavior"
        if type_ok in {"false", "incorrecto", "0", "no"}:
            return "invalid_type"
        return "invalid_major_review"

    # Pass/minor son válidos si no cayeron antes en truncamiento/error.
    if severity in {"pass", "minor"}:
        return "valid"

    # Si el modelo dijo valid true y no fue inconclusive/major/error, cuenta válido.
    if valid in {"true", "correcto", "1", "yes"}:
        return "valid"

    # Fallback.
    if valid in {"false", "incorrecto", "0", "no"}:
        return "review_not_auto_invalid"

    return "review"


# Cargar locales si hace falta
try:
    local_results
except NameError:
    with open(OUTPUT_FOLDER / "local_results.json", "r", encoding="utf-8") as f:
        local_results = json.load(f)

try:
    normalized_llm_results
except NameError:
    with open(OUTPUT_FOLDER / "llm_results.json", "r", encoding="utf-8") as f:
        normalized_llm_results = json.load(f)

all_results = local_results + normalized_llm_results
df = pd.DataFrame(all_results)

expected_columns = [
    "id",
    "dataset",
    "file_id",
    "clone_type",
    "valid",
    "reason",
    "behavior_equivalent",
    "type_compliant",
    "severity",
    "validation_source",
    "original_uses_stdin",
    "clone_uses_stdin",
    "local_sanity_warning",
]

for col in expected_columns:
    if col not in df.columns:
        df[col] = None

df["final_status"] = df.apply(final_status, axis=1)

print("Total rows:", len(df))

print("\nFinal status:")
print(df["final_status"].value_counts(dropna=False))

print("\nBy clone type:")
display(pd.crosstab(df["clone_type"], df["final_status"], margins=True))

save_csv = False  # cambia a True solo si necesitas exportar

if save_csv:
    full_csv = OUTPUT_FOLDER / "validation_results_all.csv"
    df.to_csv(full_csv, index=False)
    print("Guardado:", full_csv)

Total rows: 19996

Final status:
final_status
not_audited           10237
valid                  9536
invalid_behavior        169
invalid_type             52
needs_revalidation        2
Name: count, dtype: int64

By clone type:


final_status,invalid_behavior,invalid_type,needs_revalidation,not_audited,valid,All
clone_type,,,,,,
1,0,0,0,0,4999,4999
2,59,9,1,3492,1438,4999
3,41,4,0,3491,1463,4999
4,69,39,1,3254,1636,4999
All,169,52,2,10237,9536,19996


In [11]:
import re
import html
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# ============================================================
# 1. Filtrar los casos que la tabla cuenta como needs_revalidation
# ============================================================

needs_revalidation_df = df[df["final_status"] == "needs_revalidation"].copy()

print("Casos que necesitan revalidación:", len(needs_revalidation_df))

REVALIDATION_TERMS = [
    "truncated",
    "not enough visible evidence",
    "visible portion",
    "full assessment impossible",
    "impossible to assess",
    "impossible to verify",
]

def revalidation_trigger(row):
    severity = norm(row.get("severity"))
    reason = norm(row.get("reason"))

    if severity == "inconclusive":
        return "severity = inconclusive"

    for term in REVALIDATION_TERMS:
        if term in reason:
            return f"reason contiene: {term}"

    return "needs_revalidation por fallback/revisar lógica"

needs_revalidation_df["revalidation_trigger"] = needs_revalidation_df.apply(
    revalidation_trigger,
    axis=1
)

display(
    needs_revalidation_df[
        [
            "id",
            "dataset",
            "file_id",
            "clone_type",
            "severity",
            "validation_source",
            "final_status",
            "revalidation_trigger",
            "reason",
        ]
    ].reset_index().rename(columns={"index": "df_index"})
)

Casos que necesitan revalidación: 2


,df_index,id,dataset,file_id,clone_type,severity,validation_source,final_status,revalidation_trigger,reason
0,16172,dataset_solutions|2299_clon_2.js,dataset solutions,2299,2,major,llm,needs_revalidation,reason contiene: truncated,The clone does not preserve Type II requiremen...
1,16343,dataset_solutions|1851_clon_4.js,dataset solutions,1851,4,pass,llm,needs_revalidation,reason contiene: truncated,The clone uses a character-by-character loop t...


In [10]:
summary_rows = []

invalid_statuses = [
    "invalid_behavior",
    "invalid_type",
    "invalid_major_review",
    "invalid_local_file",
    "review_not_auto_invalid",
]

for clone_type, group in df.groupby("clone_type", dropna=False):
    total = len(group)

    valid_count = (group["final_status"] == "valid").sum()
    invalid_count = group["final_status"].isin(invalid_statuses).sum()
    needs_revalidation_count = (group["final_status"] == "needs_revalidation").sum()
    validation_error_count = (group["final_status"] == "validation_error_retry").sum()
    review_count = (group["final_status"] == "review").sum()
    not_audited_count = (group["final_status"] == "not_audited").sum()

    decisive_total = valid_count + invalid_count

    summary_rows.append({
        "clone_type": clone_type,
        "total": total,
        "valid": valid_count,
        "invalid": invalid_count,
        "needs_revalidation": needs_revalidation_count,
        "validation_error_retry": validation_error_count,
        "review": review_count,
        "not_audited": not_audited_count,
        "raw_valid_rate": valid_count / total if total else 0,
        "decisive_valid_rate": valid_count / decisive_total if decisive_total else None,
        "invalid_rate": invalid_count / total if total else 0,
        "inconclusive_rate": needs_revalidation_count / total if total else 0,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("clone_type")

overall = {
    "total": len(df),
    "valid": (df["final_status"] == "valid").sum(),
    "invalid": df["final_status"].isin(invalid_statuses).sum(),
    "needs_revalidation": (df["final_status"] == "needs_revalidation").sum(),
    "validation_error_retry": (df["final_status"] == "validation_error_retry").sum(),
    "review": (df["final_status"] == "review").sum(),
    "not_audited": (df["final_status"] == "not_audited").sum(),
}

overall["decisive_total"] = overall["valid"] + overall["invalid"]
overall["raw_valid_rate"] = overall["valid"] / overall["total"] if overall["total"] else 0
overall["decisive_valid_rate"] = (
    overall["valid"] / overall["decisive_total"]
    if overall["decisive_total"]
    else None
)

overall_df = pd.DataFrame([overall])

print("=" * 80)
print("SUMMARY BY TYPE")
print("=" * 80)
display(summary_df)

print("\n" + "=" * 80)
print("OVERALL")
print("=" * 80)
display(overall_df)

save_csv = False  # cambia a True solo si necesitas exportar los reportes

if save_csv:
    summary_csv = OUTPUT_FOLDER / "validation_summary_by_type.csv"
    overall_csv = OUTPUT_FOLDER / "validation_summary_overall.csv"

    summary_df.to_csv(summary_csv, index=False)
    overall_df.to_csv(overall_csv, index=False)

    print("\nGuardados:")
    print("Summary by type:", summary_csv)
    print("Summary overall:", overall_csv)

SUMMARY BY TYPE


,clone_type,total,valid,invalid,needs_revalidation,validation_error_retry,review,not_audited,raw_valid_rate,decisive_valid_rate,invalid_rate,inconclusive_rate
0,1,4999,4999,0,0,0,0,0,1.000000,1.000000,0.000000,0.0000
1,2,4999,1438,68,1,0,0,3492,0.287658,0.954847,0.013603,0.0002
2,3,4999,1463,45,0,0,0,3491,0.292659,0.970159,0.009002,0.0000
3,4,4999,1636,108,1,0,0,3254,0.327265,0.938073,0.021604,0.0002



OVERALL


,total,valid,invalid,needs_revalidation,validation_error_retry,review,not_audited,decisive_total,raw_valid_rate,decisive_valid_rate
0,19996,9536,221,2,0,0,10237,9757,0.476895,0.97735


In [7]:
invalid_behavior_df = df[df["final_status"] == "invalid_behavior"].copy()

for clone_type in [2, 3, 4]:
    sample = invalid_behavior_df[invalid_behavior_df["clone_type"] == clone_type].head(10)

    print("=" * 80)
    print(f"TYPE {clone_type} invalid_behavior sample")
    print("=" * 80)

    for _, r in sample.iterrows():
        print(f"- {r['id']} | reason: {r['reason']}")

TYPE 2 invalid_behavior sample
- dataset_solutions|2104_clon_2.js | reason: The clone incorrectly initializes and resets boolean flags using `!0` (which evaluates to `true`) instead of `false`, leading to a significant change in the program's behavior.
- dataset_solutions|2450_clon_2.js | reason: The clone fixes a bug present in the original's `treeAdd` function (renamed to `addVerticesToTree`), which changes the program's behavior and violates Type II requirements.
- dataset_solutions|1816_clon_2.js | reason: The content of the console.warn message changed, which is an externally visible behavior modification.
- dataset_solutions|4571_clon_2.js | reason: The debug output string content to stderr has changed, which is a violation of the behavior policy for Type II clones.
- dataset_solutions|3990_clon_2.js | reason: The clone's large number multiplication logic correctly handles carry propagation, while the original contains a critical bug in its carry calculation and propagation loop,

In [9]:
# Cell — Detectar requests del JSONL que NO aparecen en el CSV output

import json
import re
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURACIÓN
# ============================================================

INPUT_JSONL_CANDIDATES = [
    Path("validation_requests.jsonl"),
    Path("validation_request.jsonl"),
    Path("validation/validation_requests.jsonl"),
    Path("validation/validation_request.jsonl"),
]

OUTPUT_CSV_CANDIDATES = [
    Path("validation/llm_results.csv"),
    Path("llm_results.csv"),
    Path("validation/validation_results_all.csv"),
    Path("validation_results_all.csv"),
]

OUT_DIR = Path("validation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# ENCONTRAR ARCHIVOS
# ============================================================

input_jsonl_path = None
for p in INPUT_JSONL_CANDIDATES:
    if p.exists():
        input_jsonl_path = p
        break

if input_jsonl_path is None:
    print("No encontré validation_requests.jsonl. Busqué en:")
    for p in INPUT_JSONL_CANDIDATES:
        print(" -", p)
    raise FileNotFoundError("No encontré validation_requests.jsonl")

output_csv_path = None
for p in OUTPUT_CSV_CANDIDATES:
    if p.exists():
        output_csv_path = p
        break

if output_csv_path is None:
    print("No encontré CSV output. Busqué en:")
    for p in OUTPUT_CSV_CANDIDATES:
        print(" -", p)
    raise FileNotFoundError("No encontré CSV output")

print("Input JSONL:", input_jsonl_path)
print("Output CSV :", output_csv_path)

# ============================================================
# HELPERS
# ============================================================

CUSTOM_ID_RE = re.compile(
    r"^(?P<dataset>.+)\|(?P<file_id>.+)_clon_(?P<clone_type>\d+)\.js$"
)

PAIR_ID_RE = re.compile(
    r"Pair ID:\s*(?P<custom_id>[^\s]+?\|.+?_clon_\d+\.js)"
)

def normalize_dataset_name(name):
    return str(name).strip().replace(" ", "_")

def normalize_file_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    try:
        # Convierte 1816.0 -> 1816
        if re.fullmatch(r"\d+\.0", s):
            return str(int(float(s)))
    except Exception:
        pass
    return s

def normalize_clone_type(x):
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except Exception:
        return None

def parse_custom_id(custom_id):
    if custom_id is None or pd.isna(custom_id):
        return None, None, None

    custom_id = str(custom_id).strip()
    m = CUSTOM_ID_RE.match(custom_id)

    if not m:
        return None, None, None

    return (
        m.group("dataset"),
        m.group("file_id"),
        int(m.group("clone_type")),
    )

def build_custom_id(dataset, file_id, clone_type):
    if dataset is None or file_id is None or clone_type is None:
        return None
    return f"{normalize_dataset_name(dataset)}|{file_id}_clon_{int(clone_type)}.js"

def extract_prompt_text(obj):
    request = obj.get("request", obj)
    contents = request.get("contents", [])
    texts = []

    for c in contents:
        for part in c.get("parts", []):
            if "text" in part:
                texts.append(str(part["text"]))

    return "\n".join(texts)

def extract_custom_id_from_request_obj(obj):
    # Caso normal
    custom_id = obj.get("custom_id") or obj.get("id") or obj.get("request_id")
    if custom_id:
        return str(custom_id).strip()

    # Fallback: buscar Pair ID dentro del prompt
    prompt = extract_prompt_text(obj)
    m = PAIR_ID_RE.search(prompt)
    if m:
        return m.group("custom_id").strip()

    return None

def resolve_paths(dataset, file_id, clone_type):
    """
    Si DATASETS existe, construye rutas reales.
    """
    if dataset is None or file_id is None or clone_type is None:
        return None, None

    dataset_candidates = [
        str(dataset),
        str(dataset).replace("_", " "),
        str(dataset).replace(" ", "_"),
    ]

    if "DATASETS" in globals():
        for d in dataset_candidates:
            if d in DATASETS:
                original_path = Path(DATASETS[d]["js_files"]) / f"{file_id}.js"
                clone_path = Path(DATASETS[d]["clones"]) / f"{file_id}_clon_{int(clone_type)}.js"
                return str(original_path), str(clone_path)

    return None, f"{file_id}_clon_{int(clone_type)}.js"

# ============================================================
# LEER INPUT JSONL
# ============================================================

input_rows = []

with open(input_jsonl_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except Exception as e:
            input_rows.append({
                "request_line": line_num,
                "custom_id": None,
                "dataset": None,
                "file_id": None,
                "clone_type": None,
                "input_parse_error": repr(e),
            })
            continue

        custom_id = extract_custom_id_from_request_obj(obj)
        dataset, file_id, clone_type = parse_custom_id(custom_id)

        prompt = extract_prompt_text(obj)

        original_path, clone_path = resolve_paths(dataset, file_id, clone_type)

        input_rows.append({
            "request_line": line_num,
            "custom_id": custom_id,
            "dataset": dataset,
            "file_id": file_id,
            "clone_type": clone_type,
            "original_path": original_path,
            "clone_path": clone_path,
            "prompt_chars": len(prompt),
            "input_parse_error": "",
        })

input_df = pd.DataFrame(input_rows)

print("Requests en input JSONL:", len(input_df))
print("Requests sin custom_id:", input_df["custom_id"].isna().sum())

# ============================================================
# LEER OUTPUT CSV
# ============================================================

output_df = pd.read_csv(output_csv_path)

print("Filas en output CSV:", len(output_df))
print("Columnas output:")
print(output_df.columns.tolist())

# ============================================================
# EXTRAER custom_id DEL OUTPUT CSV
# ============================================================

output_df = output_df.copy()

# 1. Intentar columnas directas de ID
direct_id_cols = [
    "custom_id",
    "id",
    "request_id",
    "custom_id_norm",
    "request_custom_id",
    "prediction_custom_id",
]

id_col = None
for c in direct_id_cols:
    if c in output_df.columns:
        id_col = c
        break

if id_col is not None:
    output_df["output_custom_id"] = output_df[id_col].astype(str)
    output_df.loc[output_df["output_custom_id"].isin(["nan", "None", ""]), "output_custom_id"] = None
else:
    output_df["output_custom_id"] = None

# 2. Si no hay ID directo, construirlo desde dataset/file_id/clone_type
if output_df["output_custom_id"].isna().all():
    possible_dataset_cols = ["dataset", "parsed_dataset", "dataset_name"]
    possible_file_cols = ["file_id", "parsed_file_id"]
    possible_type_cols = ["clone_type", "parsed_clone_type"]

    dataset_col = next((c for c in possible_dataset_cols if c in output_df.columns), None)
    file_col = next((c for c in possible_file_cols if c in output_df.columns), None)
    type_col = next((c for c in possible_type_cols if c in output_df.columns), None)

    if dataset_col and file_col and type_col:
        output_df["output_custom_id"] = output_df.apply(
            lambda r: build_custom_id(
                r[dataset_col],
                normalize_file_id(r[file_col]),
                normalize_clone_type(r[type_col]),
            ),
            axis=1,
        )

# 3. Fallback: buscar custom_id dentro de texto de columnas como reason/raw/error
if output_df["output_custom_id"].isna().any():
    text_cols = [
        c for c in output_df.columns
        if any(k in c.lower() for k in ["reason", "raw", "text", "response", "error", "message"])
    ]

    def find_id_in_row_text(r):
        current = r.get("output_custom_id", None)
        if current is not None and not pd.isna(current):
            return current

        for c in text_cols:
            text = str(r.get(c, ""))
            m = re.search(r"([A-Za-z0-9_ -]+)\|(.+?_clon_\d+\.js)", text)
            if m:
                dataset = m.group(1).strip().replace(" ", "_")
                filename = m.group(2).strip()
                return f"{dataset}|{filename}"

        return None

    output_df["output_custom_id"] = output_df.apply(find_id_in_row_text, axis=1)

# Normalizar IDs
input_df["custom_id_norm"] = input_df["custom_id"].astype(str).str.strip()
input_df.loc[input_df["custom_id_norm"].isin(["nan", "None", ""]), "custom_id_norm"] = None

output_df["output_custom_id_norm"] = output_df["output_custom_id"].astype(str).str.strip()
output_df.loc[output_df["output_custom_id_norm"].isin(["nan", "None", ""]), "output_custom_id_norm"] = None

# ============================================================
# COMPARAR INPUT VS OUTPUT
# ============================================================

input_ids = set(input_df["custom_id_norm"].dropna())
output_ids = set(output_df["output_custom_id_norm"].dropna())

missing_ids = sorted(input_ids - output_ids)
extra_output_ids = sorted(output_ids - input_ids)

missing_from_output_df = input_df[
    input_df["custom_id_norm"].isin(missing_ids)
].copy()

extra_in_output_df = output_df[
    output_df["output_custom_id_norm"].isin(extra_output_ids)
].copy()

unmapped_output_df = output_df[
    output_df["output_custom_id_norm"].isna()
].copy()

# Duplicados
input_duplicates_df = input_df[
    input_df["custom_id_norm"].notna()
    & input_df["custom_id_norm"].duplicated(keep=False)
].copy()

output_duplicates_df = output_df[
    output_df["output_custom_id_norm"].notna()
    & output_df["output_custom_id_norm"].duplicated(keep=False)
].copy()

# ============================================================
# GUARDAR REPORTES OPCIONALMENTE
# ============================================================

save_reports = False  # cambia a True solo cuando quieras exportar archivos

summary = {
    "input_jsonl": str(input_jsonl_path),
    "output_csv": str(output_csv_path),
    "input_requests": int(len(input_df)),
    "input_unique_ids": int(len(input_ids)),
    "output_rows": int(len(output_df)),
    "output_unique_ids": int(len(output_ids)),
    "missing_from_output": int(len(missing_from_output_df)),
    "extra_in_output": int(len(extra_in_output_df)),
    "unmapped_output_rows": int(len(unmapped_output_df)),
    "input_duplicate_rows": int(len(input_duplicates_df)),
    "output_duplicate_rows": int(len(output_duplicates_df)),
}

if save_reports:
    reports_dir = OUT_DIR / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)

    missing_csv = reports_dir / "requests_in_jsonl_missing_from_output_csv.csv"
    extra_csv = reports_dir / "rows_in_output_not_in_input_jsonl.csv"
    unmapped_csv = reports_dir / "rows_in_output_without_custom_id.csv"
    input_dupes_csv = reports_dir / "duplicate_requests_in_input_jsonl.csv"
    output_dupes_csv = reports_dir / "duplicate_rows_in_output_csv.csv"
    summary_json = reports_dir / "input_vs_output_comparison_summary.json"

    missing_from_output_df.to_csv(missing_csv, index=False)
    extra_in_output_df.to_csv(extra_csv, index=False)
    unmapped_output_df.to_csv(unmapped_csv, index=False)
    input_duplicates_df.to_csv(input_dupes_csv, index=False)
    output_duplicates_df.to_csv(output_dupes_csv, index=False)

    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nReportes guardados en:", reports_dir)
else:
    print("\nReportes no guardados. Cambia save_reports = True para exportarlos.")

# ============================================================
# MOSTRAR RESUMEN
# ============================================================

print("=" * 80)
print("COMPARACIÓN INPUT JSONL VS OUTPUT CSV")
print("=" * 80)
print("Requests input JSONL       :", len(input_df))
print("IDs únicos input           :", len(input_ids))
print("Filas output CSV           :", len(output_df))
print("IDs únicos output          :", len(output_ids))
print("INPUT que NO están en CSV  :", len(missing_from_output_df))
print("OUTPUT sin custom_id       :", len(unmapped_output_df))
print("OUTPUT extra no en input   :", len(extra_in_output_df))
print("Duplicados input           :", len(input_duplicates_df))
print("Duplicados output          :", len(output_duplicates_df))

print("\nReportes:")
print("Faltan del output :", missing_csv)
print("Output sin ID     :", unmapped_csv)
print("Extra en output   :", extra_csv)
print("Dupes input       :", input_dupes_csv)
print("Dupes output      :", output_dupes_csv)
print("Summary           :", summary_json)

if len(missing_from_output_df):
    print("\nEstos requests están en el JSONL pero NO aparecen en el CSV:")
    display(missing_from_output_df[
        [
            "request_line",
            "custom_id",
            "dataset",
            "file_id",
            "clone_type",
            "original_path",
            "clone_path",
            "prompt_chars",
        ]
    ].head(50))

if len(unmapped_output_df):
    print("\nFilas del output que no tienen custom_id detectable:")
    display(unmapped_output_df.head(20))

Input JSONL: validation_requests.jsonl
Output CSV : validation/llm_results.csv
Requests en input JSONL: 4760
Requests sin custom_id: 0
Filas en output CSV: 4760
Columnas output:
['id', 'dataset', 'file_id', 'clone_type', 'valid', 'reason', 'validation_source', 'behavior_equivalent', 'type_compliant', 'severity', 'original_uses_stdin', 'clone_uses_stdin', 'local_sanity_warning']

Reportes no guardados. Cambia save_reports = True para exportarlos.
COMPARACIÓN INPUT JSONL VS OUTPUT CSV
Requests input JSONL       : 4760
IDs únicos input           : 4760
Filas output CSV           : 4760
IDs únicos output          : 4760
INPUT que NO están en CSV  : 0
OUTPUT sin custom_id       : 0
OUTPUT extra no en input   : 0
Duplicados input           : 0
Duplicados output          : 0

Reportes:
Faltan del output : validation/requests_in_jsonl_missing_from_output_csv.csv
Output sin ID     : validation/rows_in_output_without_custom_id.csv
Extra en output   : validation/rows_in_output_not_in_input_jsonl.